In [1]:
from datetime import datetime
from pathlib import Path

import xarray as xr

# Funciona tanto si Jupyter se inicia en la raíz del proyecto como en notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
satellite_repository = "/mnt/Datos/ESP_SAT"

satellite_repository = Path(satellite_repository)
netcdf_files = sorted(satellite_repository.rglob("*.nc"))
if not netcdf_files:
    raise FileNotFoundError(f"No se encontraron archivos .nc en {satellite_repository}")

print(f"Archivos encontrados: {len(netcdf_files)}")
netcdf_files


Archivos encontrados: 46383


[PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0000.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0015.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0030.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0045.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0100.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0115.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0130.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0145.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0200.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0215.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0230.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0245.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0300.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0315.nc'),
 PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_

In [2]:
netcdf_files[0].parts

('/', 'mnt', 'Datos', 'ESP_SAT', '2020', '01', 'HRSEVIRI_20200101T0000.nc')

In [3]:
def add_time_from_filename(ds):
    # Ejemplo: HRSEVIRI_20191231T2215.nc → 2019-12-31 22:15.
    timestamp = Path(ds.encoding["source"]).stem.rsplit("_", 1)[-1]
    return ds.expand_dims(time=[datetime.strptime(timestamp, "%Y%m%dT%H%M")])


netcdf_files_2020 =  [f for f in netcdf_files if "2020" in f.parts]

netcdf_files_2020_jan = [f for f in netcdf_files_2020 if "01" in f.parts]

netcdf_files_2020_jan, len(netcdf_files_2020_jan), netcdf_files_2020_jan[-5:]

([PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0000.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0015.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0030.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0045.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0100.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0115.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0130.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0145.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0200.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0215.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0230.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0245.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0300.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/2020/01/HRSEVIRI_20200101T0315.nc'),
  PosixPath('/mnt/Datos/ESP_SAT/20

In [4]:

#dataset_2020_jan = xr.open_mfdataset(
    #netcdf_files_2020_jan,
    #engine="h5netcdf",
    #combine="by_coords",
    #preprocess=add_time_from_filename,
    #chunks={"lat": 301, "lon": 458},
#)
#dataset_2020_jan


In [5]:
# Save the netcdf dataset grouped by 3 months to 3 months in zarr 

from glob import glob

def _select_months_for_quarter(quarter):
    if quarter == 1:
        return [1, 2, 3]
    elif quarter == 2:
        return [4, 5, 6]
    elif quarter == 3:
        return [7, 8, 9]
    elif quarter == 4:
        return [10, 11, 12]
    else:
        raise ValueError("Quarter must be between 1 and 4")

def load_netcdfs_by_quarters_lazy(file_list, year, quarter):
    months = {f"{month:02d}" for month in _select_months_for_quarter(quarter)}

    files = sorted(
        f for f in file_list
        if f.parent.name in months and f.parent.parent.name == str(year)
    )

    if not files:
        raise ValueError(f"No hay archivos para {year}, trimestre {quarter}")

    print(f"Archivos seleccionados: {len(files)}")
    print(f"Primero: {files[0].name}")
    print(f"Último: {files[-1].name}")

    return xr.open_mfdataset(
        files,
        engine="h5netcdf",
        combine="by_coords",
        preprocess=add_time_from_filename,
        chunks={"lat": 301, "lon": 458},
    )

def save_dataset_by_quarters(dataset, root):
    filename = f"dataset_{dataset.time.dt.year.values[0]}_q{((dataset.time.dt.month.values[0]-1)//3)+1}.zarr"
    path = root / filename
    dataset.to_zarr(path)
    print(f"Dataset guardado en: {path}")

dataset_2020_q4 = load_netcdfs_by_quarters_lazy(netcdf_files, 2020, 4)
save_dataset_by_quarters(dataset_2020_q4, Path("/home/marito/Documents/Projects/forest-fire-detection/data/satellite/ESP_PROCESS"))

Archivos seleccionados: 2205
Primero: HRSEVIRI_20201001T0000.nc
Último: HRSEVIRI_20201231T2300.nc


/home/marito/Documents/Projects/forest-fire-detection/.venv/lib/python3.13/site-packages/zarr/core/dtype/npy/bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/marito/Documents/Projects/forest-fire-detection/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Dataset guardado en: /home/marito/Documents/Projects/forest-fire-detection/data/satellite/ESP_PROCESS/dataset_2020_q4.zarr
